# 🧬 Módulo 4: Modelado de Proteínas y Docking Molecular
## Actividad 4.7: Docking Proteína-Proteína

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_04_modelado_proteinas_docking/07_docking_proteina_proteina.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Comprender la complejidad de interfaces proteína-proteína
- Usar herramientas especializadas (HADDOCK, ClusPro)
- Predecir sitios de unión en superficies proteicas
- Analizar interfaces de contacto
- Evaluar energías de interacción
- Validar complejos predichos

---

## 📚 Introducción

El docking proteína-proteína es más complejo que el docking de ligandos pequeños debido al tamaño de las moléculas y la flexibilidad de las interfaces.

---

In [ ]:
# Instalación de dependencias
!pip install biopython requests py3Dmol numpy pandas matplotlib scipy
print("✓ Dependencias instaladas")

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import py3Dmol
from Bio import PDB
from Bio.PDB import PDBParser, PDBIO, Select, NeighborSearch
from Bio.PDB.Polypeptide import PPBuilder
from scipy.spatial.distance import cdist
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas")

## 📚 Introducción al Docking Proteína-Proteína

Las interacciones proteína-proteína (PPIs) son fundamentales en:
- Señalización celular
- Formación de complejos enzimáticos
- Regulación génica
- Ensamblaje viral

### Diferencias con el docking ligando-proteína

| Aspecto | Ligando-Proteína | Proteína-Proteína |
|---------|-----------------|-------------------|
| **Tamaño del ligando** | 200-600 Da | 10,000-100,000 Da |
| **Área de interfaz** | 200-500 Å² | 700-4,000 Å² |
| **Flexibilidad** | Un componente flexible | Dos proteínas flexibles |
| **Complejidad** | ~10-20 grados de libertad | ~100+ grados de libertad |
| **Herramientas** | AutoDock Vina, Glide | HADDOCK, ClusPro, RosettaDock |
| **Tiempo de cálculo** | Minutos | Horas-días |

### Tipos de complejos

1. **Homodímeros**: Dos moléculas idénticas
2. **Heterodímeros**: Dos proteínas distintas (enzima-inhibidor)
3. **Complejos transitorios**: Señalización (quinasa-sustrato)
4. **Complejos permanentes**: Estructurales (hemoglobina)

## 1. Análisis de Interfaces en Complejos Conocidos

Trabajaremos con el complejo **tripsina-inhibidor de Kunitz** (PDB: **2PTC**), un sistema clásico de interacción enzima-inhibidor proteico.

In [ ]:
Path("ppi_analisis").mkdir(exist_ok=True)

def descargar_pdb(pdb_id, output_dir="ppi_analisis"):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url)
    out = Path(output_dir) / f"{pdb_id}.pdb"
    if r.status_code == 200:
        out.write_text(r.text)
        print(f"✓ {pdb_id}.pdb descargado")
        return out
    print(f"✗ Error descargando {pdb_id}")
    return None

def analizar_interfaz_ppi(pdb_file, cadena1='E', cadena2='I', umbral_distancia=5.0):
    """
    Analiza la interfaz de contacto entre dos cadenas proteicas.
    
    Identifica residuos de interfaz: aquellos con al menos un átomo
    a menos de 'umbral_distancia' Å de la otra cadena.
    """
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("complejo", pdb_file)
    
    # Obtener átomos de cada cadena
    atomos_c1, atomos_c2 = [], []
    res_c1, res_c2 = {}, {}
    
    for model in struct:
        for chain in model:
            for residue in chain:
                if residue.id[0] != ' ':
                    continue
                for atom in residue:
                    coord = atom.coord
                    key = f"{residue.resname}{residue.id[1]}"
                    if chain.id == cadena1:
                        atomos_c1.append(coord)
                        res_c1[key] = residue
                    elif chain.id == cadena2:
                        atomos_c2.append(coord)
                        res_c2[key] = residue
    
    if not atomos_c1 or not atomos_c2:
        print(f"⚠️  No se encontraron cadenas {cadena1} o {cadena2}")
        return {}, {}
    
    # Matriz de distancias entre todas las combinaciones de átomos
    mat_dist = cdist(np.array(atomos_c1), np.array(atomos_c2))
    
    # Identificar pares de átomos en contacto
    indices_c1, indices_c2 = np.where(mat_dist < umbral_distancia)
    
    print(f"\n{'='*50}")
    print(f"  ANÁLISIS DE INTERFAZ: Cadena {cadena1} — Cadena {cadena2}")
    print(f"  Umbral de distancia: {umbral_distancia} Å")
    print(f"{'='*50}")
    print(f"  Átomos cadena {cadena1}: {len(atomos_c1)}")
    print(f"  Átomos cadena {cadena2}: {len(atomos_c2)}")
    print(f"  Contactos inter-cadena: {len(indices_c1)}")
    print(f"  Distancia mínima: {mat_dist.min():.2f} Å")
    print(f"{'='*50}")
    
    return mat_dist, (indices_c1, indices_c2)

# Descargar complejo Tripsina-BPTI (2PTC)
pdb_2ptc = descargar_pdb("2PTC")
if pdb_2ptc:
    mat_dist, contactos = analizar_interfaz_ppi(str(pdb_2ptc), cadena1='E', cadena2='I')

## 2. Identificación de Residuos de Interfaz y Hotspots

Los **hotspots** son residuos en la interfaz que contribuyen de manera desproporcionada a la energía de unión (típicamente >2 kcal/mol cada uno). Representan solo el 20% de los residuos de interfaz, pero el 80% de la energía de unión.

Características de hotspots:
- **Aromáticos**: Tyr, Trp, Phe — interacciones π-π y empaquetamiento
- **Cargados**: Arg, Glu, Asp — sal bridges y H-bonds
- **Hidrofóbicos expuestos**: Leu, Ile, Val

In [ ]:
def identificar_residuos_interfaz(pdb_file, cadena1='E', cadena2='I', umbral=5.0):
    """
    Identifica los residuos en la interfaz de dos cadenas proteicas.
    Calcula el área de superficie accesible (ASA) aproximada.
    """
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("complejo", pdb_file)
    
    # Recopilar átomos por cadena
    info_cadenas = {cadena1: {}, cadena2: {}}
    
    for model in struct:
        for chain in model:
            if chain.id not in info_cadenas:
                continue
            for residue in chain:
                if residue.id[0] != ' ':
                    continue
                key = (chain.id, residue.id[1], residue.resname)
                atomos = [a.coord for a in residue]
                info_cadenas[chain.id][key] = atomos
    
    # Encontrar residuos de interfaz
    residuos_interfaz = {cadena1: [], cadena2: []}
    
    for key1, atomos1 in info_cadenas[cadena1].items():
        for key2, atomos2 in info_cadenas[cadena2].items():
            mat = cdist(np.array(atomos1), np.array(atomos2))
            if mat.min() < umbral:
                if key1 not in residuos_interfaz[cadena1]:
                    residuos_interfaz[cadena1].append(key1)
                if key2 not in residuos_interfaz[cadena2]:
                    residuos_interfaz[cadena2].append(key2)
    
    # Clasificar residuos de interfaz
    aa_cargados = ['ARG','LYS','ASP','GLU','HIS']
    aa_aromaticos = ['TYR','TRP','PHE']
    aa_hidrofobicos = ['LEU','ILE','VAL','MET','ALA','PRO']
    
    for cadena_id, residuos in residuos_interfaz.items():
        print(f"\nResiduos de interfaz en Cadena {cadena_id} ({len(residuos)}):")
        conteo = {'Cargados': 0, 'Aromáticos': 0, 'Hidrofóbicos': 0, 'Otros': 0}
        for (cid, rnum, rname) in residuos[:10]:
            tipo = ('Cargado' if rname in aa_cargados else
                    'Aromático' if rname in aa_aromaticos else
                    'Hidrofóbico' if rname in aa_hidrofobicos else 'Otro')
            key = tipo + 's'
            if key in conteo:
                conteo[key] += 1
            print(f"  {rname:3s} {rnum:4d}  [{tipo}]")
        
        if len(residuos) > 10:
            print(f"  ... y {len(residuos)-10} más")
    
    return residuos_interfaz

# Identificar residuos de interfaz en el complejo tripsina-BPTI
if pdb_2ptc:
    residuos_if = identificar_residuos_interfaz(str(pdb_2ptc))

## 3. Herramientas de Docking Proteína-Proteína

### 3.1 HADDOCK (High Ambiguity Driven protein-protein DOCKing)

HADDOCK utiliza **restricciones ambiguas de interacción (AIRs)** derivadas de datos experimentales o predicciones:
- Datos de NMR (chemical shifts)
- Mutagénesis dirigida
- Predicciones de sitios de unión

**Flujo de trabajo HADDOCK:**
```
Datos biológicos → AIRs → Docking rígido → Semi-flexible → Refinamiento en agua
```

### 3.2 ClusPro (ZDOCK + clustering)

ClusPro usa transformaciones de Fourier para explorar el espacio de docking y agrupa las poses por similitud geométrica.

### 3.3 RosettaDock

Parte de la suite Rosetta; optimiza tanto la geometría como la energía con alta precisión.

| Herramienta | Acceso | Velocidad | Restricciones |
|-------------|--------|-----------|---------------|
| **HADDOCK** | Web/local | Media | Requiere AIRs |
| **ClusPro** | Web gratuito | Rápida | No requeridas |
| **RosettaDock** | Local | Lenta | No requeridas |
| **ZDOCK** | Web/local | Rápida | No requeridas |

In [ ]:
def preparar_input_haddock(pdb1, pdb2, residuos_activos_1, residuos_activos_2, 
                            output_dir="ppi_analisis"):
    """
    Genera el archivo de restricciones AIR para HADDOCK.
    Los residuos activos son los hotspots de interfaz.
    """
    # Formato de restricciones ambiguas HADDOCK (CNS format)
    restricciones = []
    
    for res1 in residuos_activos_1[:5]:  # Usamos primeros 5 como ejemplo
        for res2 in residuos_activos_2[:5]:
            restriccion = (
                f"assign (segid A and resid {res1})\n"
                f"       (segid B and resid {res2}) 2.0 2.0 0.0"
            )
            restricciones.append(restriccion)
    
    # Generar archivo de restricciones
    air_content = "\n!AIR restraints generated for HADDOCK\n\n"
    air_content += "\n\n".join(restricciones)
    
    air_file = Path(output_dir) / "air_restraints.tbl"
    air_file.write_text(air_content)
    print(f"✓ Restricciones AIR guardadas: {air_file}")
    
    # Mostrar parámetros típicos de HADDOCK
    print("\n📋 Parámetros típicos para HADDOCK web server:")
    print("="*50)
    print(f"  Proteína 1: {pdb1}")
    print(f"  Proteína 2: {pdb2}")
    print(f"  Residuos activos (prot 1): {residuos_activos_1[:5]}")
    print(f"  Residuos activos (prot 2): {residuos_activos_2[:5]}")
    print(f"  it0 (rigid):      1000 estructuras")
    print(f"  it1 (semiflex):   200  estructuras")
    print(f"  water (refin):    200  estructuras")
    print(f"  Clustering:       RMSD, umbral 7.5 Å")
    print("="*50)
    
    return air_file

# Ejemplo de residuos de interfaz para BPTI-Tripsina
residuos_tripsina = [35, 57, 190, 195, 213, 216]  # Sitio activo tripsina
residuos_bpti     = [15, 16, 17, 18, 19, 34, 38]  # Región de contacto BPTI

air = preparar_input_haddock(
    "2PTC_chainE.pdb", "2PTC_chainI.pdb",
    residuos_tripsina, residuos_bpti
)

## 4. Visualización de la Interfaz Proteína-Proteína

In [ ]:
def visualizar_complejo_ppi(pdb_file, cadena1='E', cadena2='I',
                             res_interfaz_c1=None, res_interfaz_c2=None):
    """
    Visualiza el complejo proteína-proteína con las dos cadenas
    coloreadas diferente y los residuos de interfaz resaltados.
    """
    with open(pdb_file, 'r') as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=900, height=600)
    view.addModel(pdb_data, 'pdb')
    
    # Cadena 1: azul (proteína receptora)
    view.setStyle({'chain': cadena1},
                 {'cartoon': {'color': '#4488cc', 'opacity': 0.85}})
    
    # Cadena 2: rojo (proteína ligando)
    view.setStyle({'chain': cadena2},
                 {'cartoon': {'color': '#cc4444', 'opacity': 0.85}})
    
    # Residuos de interfaz de cadena 1: sticks amarillos
    if res_interfaz_c1:
        for res_id in res_interfaz_c1:
            view.addStyle({'chain': cadena1, 'resi': res_id},
                         {'stick': {'colorscheme': 'yellowCarbon', 'radius': 0.25}})
    
    # Residuos de interfaz de cadena 2: sticks naranjas
    if res_interfaz_c2:
        for res_id in res_interfaz_c2:
            view.addStyle({'chain': cadena2, 'resi': res_id},
                         {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.25}})
    
    view.zoomTo()
    view.setBackgroundColor('#1a1a2e')
    
    print("Azul: Tripsina (cadena E) | Rojo: BPTI-Inhibidor (cadena I)")
    print("Amarillo: Residuos interfaz tripsina | Naranja: Residuos interfaz BPTI")
    return view

# Visualizar complejo
if pdb_2ptc:
    view = visualizar_complejo_ppi(
        str(pdb_2ptc), 
        cadena1='E', cadena2='I',
        res_interfaz_c1=residuos_tripsina,
        res_interfaz_c2=residuos_bpti
    )
    view.show()

## 5. Análisis Estadístico de la Interfaz

Una buena interfaz PPI tiene características geométricas y energéticas específicas.

In [ ]:
def analizar_estadisticas_interfaz(mat_dist, umbral=5.0):
    """
    Calcula estadísticas de la interfaz proteína-proteína.
    """
    if mat_dist is None or len(mat_dist) == 0:
        print("No hay datos de matriz de distancias.")
        return
    
    # Métricas de la interfaz
    contactos = mat_dist < umbral
    n_contactos = contactos.sum()
    dist_minima = mat_dist.min()
    
    # Distribución de distancias de contacto
    dists_contacto = mat_dist[contactos]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Histograma de distancias de contacto
    axes[0].hist(dists_contacto, bins=25, color='steelblue', 
                edgecolor='white', alpha=0.85)
    axes[0].axvline(dist_minima, color='red', linestyle='--',
                   label=f'Mínimo: {dist_minima:.2f} Å')
    axes[0].axvline(3.5, color='orange', linestyle='--', alpha=0.7,
                   label='VdW típico: 3.5 Å')
    axes[0].set_xlabel('Distancia (Å)')
    axes[0].set_ylabel('Número de contactos')
    axes[0].set_title('Distribución de Distancias\nen la Interfaz')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Mapa de contactos (subsample para visualización)
    n = min(100, mat_dist.shape[0])
    m = min(100, mat_dist.shape[1])
    sub_mat = mat_dist[:n, :m]
    im = axes[1].imshow(sub_mat < umbral, cmap='Blues', aspect='auto')
    axes[1].set_xlabel(f'Átomos Cadena 2 (primeros {m})')
    axes[1].set_ylabel(f'Átomos Cadena 1 (primeros {n})')
    axes[1].set_title(f'Mapa de Contactos (d < {umbral} Å)')
    plt.colorbar(im, ax=axes[1], label='Contacto')
    
    # Estadísticas comparativas con valores de referencia
    stats = {
        'Complejidad\nde la interfaz': (n_contactos, 500, 2000, 'Buena'),
        'Área de interfaz\n(estimada)': (n_contactos * 15, 700, 3000, 'Buena'),
    }
    
    categorias = [f'{k}\n({v[0]:,})' for k,v in stats.items()]
    valores = [min(v[0]/v[2], 1.2) for v in stats.values()]
    colores_bar = ['green' if v[1] <= v[0] <= v[2] else 'orange' 
                  for v in stats.values()]
    
    axes[2].barh(categorias, valores, color=colores_bar, alpha=0.8)
    axes[2].axvline(1.0, color='red', linestyle='--', label='Máximo esperado')
    axes[2].set_xlabel('Fracción del valor esperado')
    axes[2].set_title('Evaluación de la Interfaz')
    axes[2].grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Estadísticas de Interfaz — Tripsina/BPTI (2PTC)', 
                fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 RESUMEN DE LA INTERFAZ:")
    print(f"  Contactos totales (d < {umbral} Å): {n_contactos}")
    print(f"  Distancia mínima: {dist_minima:.2f} Å")
    print(f"  Distancia promedio (contactos): {dists_contacto.mean():.2f} Å")

if isinstance(mat_dist, np.ndarray) and mat_dist.size > 0:
    analizar_estadisticas_interfaz(mat_dist)

## 6. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Descarga el complejo **antígeno-anticuerpo** (PDB: **1MHP**):
1. Identifica las dos cadenas del complejo
2. Cuenta los residuos de interfaz de cada cadena (umbral 5 Å)
3. ¿Cuántos contactos hay en total?

### Ejercicio 2 (Intermedio)
Con el complejo **SH3-peptido** (PDB: **1PRM**):
1. Analiza la interfaz y clasifica los residuos por tipo de aminoácido
2. ¿Qué tipo predomina en la interfaz — hidrofóbicos o cargados?
3. Visualiza el complejo con Py3Dmol resaltando la interfaz

### Ejercicio 3 (Avanzado)
Usa el servidor **HADDOCK** (https://wenmr.science.uu.nl/haddock2.4/):
1. Separa las cadenas A y B de **2PTC** en archivos PDB independientes
2. Usa los residuos de interfaz identificados como AIRs
3. Ejecuta el docking y compara las poses predichas con el complejo experimental
4. Calcula el RMSD entre la mejor pose y la estructura 2PTC

In [ ]:
# Ejercicio 1: Complejo antígeno-anticuerpo (1MHP)
pdb_1mhp = descargar_pdb("1MHP")
if pdb_1mhp:
    _ = analizar_interfaz_ppi(str(pdb_1mhp), cadena1='L', cadena2='H')
# Tu código aquí para los ejercicios 2 y 3...

## 7. Referencias

1. Dominguez, C. et al. (2003). HADDOCK: a protein-protein docking approach based on biochemical or biophysical information. *J. Am. Chem. Soc.*, 125, 1731-1737.
2. Kozakov, D. et al. (2017). The ClusPro web server for protein-protein docking. *Nature Protocols*, 12, 255-278.
3. Janin, J. & Wodak, S.J. (1978). Conformation of amino acid side-chains at protein surfaces. *J. Mol. Biol.*, 125, 357-386.
4. Berman, H.M. et al. (2000). The Protein Data Bank. *Nucleic Acids Res.*, 28, 235-242.
5. Lo Conte, L. et al. (1999). The atomic structure of protein-protein recognition sites. *J. Mol. Biol.*, 285, 2177-2198.

---

## 📚 Recursos Adicionales

### Servidores Web
- [HADDOCK 2.4](https://wenmr.science.uu.nl/haddock2.4/) — Docking guiado por restricciones
- [ClusPro](https://cluspro.bu.edu/) — Docking automático por clustering
- [ZDOCK](https://zdock.umassmed.edu/) — Fast Fourier Transform docking
- [PatchDock](https://bioinfo3d.cs.tau.ac.il/PatchDock/) — Docking basado en geometría

### Análisis de Interfaces
- [PDBePISA](https://www.ebi.ac.uk/pdbe/pisa/) — Análisis de interfaces, ASA y energías
- [COCOMAPS](https://www.molnac.unisa.it/BioTools/cocomaps/) — Mapas de contacto
- [InterProt](http://interprotdb.jp/) — Base de datos de interacciones

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Explicar las diferencias entre docking ligando-proteína y proteína-proteína
- ✅ Identificar residuos de interfaz y hotspots usando BioPython
- ✅ Preparar restricciones AIR para HADDOCK
- ✅ Interpretar mapas de contacto y estadísticas de interfaz
- ✅ Usar servidores web de docking PPI (HADDOCK, ClusPro)
- ✅ Visualizar complejos PPI con Py3Dmol

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 4.7: Docking Proteína-Proteína**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_4.6-Docking_Ligando_Proteína-blue.svg)](06_docking_ligando_proteina.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_4.8_➡️-Virtual_Screening-green.svg)](08_virtual_screening.ipynb)

---

📚 **[Volver al Módulo 4](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>